In [ ]:
import os
from pathlib import Path
import pandas as pd
import shutil
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from PIL import Image

# 1. Define Directories and Unzip Dataset
DATA_DIR = Path("data")  # Adjust this path to where your zip files and CSVs are located
DATASET_DIR = Path("datasets/dataset")
IMAGES_DIR = DATASET_DIR / "images"
TRAIN_IMAGES_DIR = IMAGES_DIR / "train"
VAL_IMAGES_DIR = IMAGES_DIR / "val"
TEST_IMAGES_DIR = IMAGES_DIR / "test"

LABELS_DIR = DATASET_DIR / "labels"
TRAIN_LABELS_DIR = LABELS_DIR / "train"
VAL_LABELS_DIR = LABELS_DIR / "val"
TEST_LABELS_DIR = LABELS_DIR / "test"

# Unzip the images
shutil.unpack_archive(DATA_DIR / "images.zip", IMAGES_DIR)

# Load train and test CSV files
train = pd.read_csv(DATA_DIR / "Train.csv")
test = pd.read_csv(DATA_DIR / "Test.csv")
sample_submission = pd.read_csv(DATA_DIR / "SampleSubmission.csv")

# Add image paths to the DataFrame
train["image_path"] = [IMAGES_DIR / x for x in train["Image_ID"]]
test["image_path"] = [IMAGES_DIR / x for x in test["Image_ID"]]

# Map class names to integers
class_mapper = {cls: idx for idx, cls in enumerate(sorted(train["class"].unique()))}
train["class_id"] = train["class"].map(class_mapper)


In [ ]:

# 2. Split Data into Train and Validation
train_unique_imgs_df = train.drop_duplicates(subset=["Image_ID"], ignore_index=True)
X_train, X_val = train_test_split(
    train_unique_imgs_df, test_size=0.25, stratify=train_unique_imgs_df["class"], random_state=42
)

X_train = train[train["Image_ID"].isin(X_train["Image_ID"])]
X_val = train[train["Image_ID"].isin(X_val["Image_ID"])]

# Create directories
for DIR in [TRAIN_IMAGES_DIR, VAL_IMAGES_DIR, TEST_IMAGES_DIR, TRAIN_LABELS_DIR, VAL_LABELS_DIR, TEST_LABELS_DIR]:
    if DIR.exists():
        shutil.rmtree(DIR)
    DIR.mkdir(parents=True, exist_ok=True)

# Copy train, validation, and test images
for img in tqdm(X_train["image_path"].unique(), desc="Copying Train Images"):
    shutil.copy(img, TRAIN_IMAGES_DIR / img.name)

for img in tqdm(X_val["image_path"].unique(), desc="Copying Validation Images"):
    shutil.copy(img, VAL_IMAGES_DIR / img.name)

for img in tqdm(test["image_path"].unique(), desc="Copying Test Images"):
    shutil.copy(img, TEST_IMAGES_DIR / img.name)

# 3. Define Dataset Class
class CustomDataset(Dataset):
    def __init__(self, dataframe, root_dir, transforms=None):
        self.data = dataframe
        self.root_dir = root_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        record = self.data.iloc[idx]
        image_path = record["image_path"]
        image = Image.open(image_path).convert("RGB")

        # Get bounding boxes and labels
        boxes = record[["x_min", "y_min", "x_max", "y_max"]].values.reshape(-1, 4)
        labels = [record["class_id"]]

        # Convert to tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            image = self.transforms(image)

        return image, target

# 4. Prepare DataLoaders
train_dataset = CustomDataset(X_train, TRAIN_IMAGES_DIR)
val_dataset = CustomDataset(X_val, VAL_IMAGES_DIR)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# 5. Define Faster R-CNN Model
def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

num_classes = train["class_id"].nunique() + 1  # Number of unique classes + 1 for background
model = get_model(num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 6. Training Loop
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for images, targets in train_loader:
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        epoch_loss += losses.item()

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    lr_scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss}")

torch.save(model.state_dict(), "fasterrcnn_model.pth")
print("Model saved as fasterrcnn_model.pth.")

# 7. Generate Submission File
test_dataset = CustomDataset(test, TEST_IMAGES_DIR)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

model.eval()
predictions = []
with torch.no_grad():
    for images, targets in test_loader:
        images = [image.to(device) for image in images]
        outputs = model(images)

        for i, output in enumerate(outputs):
            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            labels = output["labels"].cpu().numpy()
            image_id = test.iloc[i]["Image_ID"]

            for box, score, label in zip(boxes, scores, labels):
                if score > 0.5:
                    predictions.append({
                        "Image_ID": image_id,
                        "x_min": box[0],
                        "y_min": box[1],
                        "x_max": box[2],
                        "y_max": box[3],
                        "score": score,
                        "class": label
                    })

submission_df = pd.DataFrame(predictions)
submission_df.to_csv("submission.csv", index=False)
print("Submission file saved as submission.csv.")
